In [1]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from pathlib import Path

ruta = Path("../Datos/procesados")

# Cargar mercados
mercados = gpd.read_parquet(
    ruta / "mercados_cdmx_sedeco.parquet"
)

# Cargar tianguis
tianguis = pd.read_parquet(
    ruta / "tianguis_cdmx_sedeco_2024.parquet"
)

# Cargar DENUE
denue = pd.read_parquet(
    ruta / "denue_4611_cdmx_2026.parquet"
)

print("Mercados:", len(mercados))
print("Tianguis:", len(tianguis))
print("DENUE:", len(denue))
print("CRS mercados:", mercados.crs)

Mercados: 329
Tianguis: 1334
DENUE: 86931
CRS mercados: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "ITRF92", "datum": {"type": "DynamicGeodeticReferenceFrame", "name": "International Terrestrial Reference Frame 1992", "frame_reference_epoch": 1988, "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Geodesy.", "area": "World.", "bbox": {"south_latitude": -90, "west_longitude": -180, "north_latitude": 90, "east_longitude": 180}, "id": {"authority": "EPSG", "code": 8992}}


In [2]:
# Convertir geometrías de mercados a WGS84
mercados = mercados.to_crs(epsg=4326)

# Actualizar coordenadas a partir de la geometría transformada
mercados["latitud_mapa"] = mercados.geometry.y
mercados["longitud_mapa"] = mercados.geometry.x

# Tianguis con coordenadas válidas
tianguis_mapa = tianguis[
    tianguis["latitud"].between(19.0, 19.7) &
    tianguis["longitud"].between(-99.4, -98.8) &
    (tianguis["estado"] != "NO_SE_INSTALA")
].copy()

print("Mercados para mapa:", len(mercados))
print("Tianguis para mapa:", len(tianguis_mapa))

Mercados para mapa: 329
Tianguis para mapa: 1332


In [4]:
from pathlib import Path

ruta_mapa = Path("../app/mapa_comercio.html")
ruta_mapa.parent.mkdir(parents=True, exist_ok=True)

mapa.save(str(ruta_mapa))

print("Mapa guardado en:", ruta_mapa.resolve())

Mapa guardado en: C:\Users\LENOVO\Documents\chilangotea\Chilangaton\app\mapa_comercio.html


In [5]:
from folium.plugins import HeatMap

# Seleccionar establecimientos con coordenadas válidas
denue_mapa = denue[
    denue["latitud"].between(19.0, 19.7) &
    denue["longitud"].between(-99.4, -98.8)
].copy()

print("Establecimientos para el mapa:", len(denue_mapa))

# Crear capa de densidad comercial
grupo_denue = folium.FeatureGroup(
    name="DENUE - Densidad comercial",
    show=False
)

HeatMap(
    denue_mapa[["latitud", "longitud"]].values.tolist(),
    radius=12,
    blur=15,
    min_opacity=0.3
).add_to(grupo_denue)

grupo_denue.add_to(mapa)

Establecimientos para el mapa: 86931


In [6]:
folium.LayerControl().add_to(mapa)

In [7]:
folium.LayerControl().add_to(mapa)

mapa.save("../app/mapa_comercio.html")

print("Mapa actualizado correctamente")

Mapa actualizado correctamente


In [9]:
print("Capas registradas:")
for elemento in mapa._children.values():
    if isinstance(elemento, folium.map.FeatureGroup):
        print(elemento.layer_name)

Capas registradas:
DENUE - Densidad comercial


In [10]:
print("CAPAS REGISTRADAS")

for elemento in mapa._children.values():
    if isinstance(elemento, (folium.FeatureGroup, folium.plugins.MarkerCluster)):
        print(elemento.layer_name)

CAPAS REGISTRADAS
Mercados públicos
Tianguis
DENUE - Densidad comercial


In [11]:
print("Mercados:", len(grupo_mercados._children))
print("Tianguis:", len(grupo_tianguis._children))
print("DENUE:", len(denue_mapa))

Mercados: 330
Tianguis: 1333
DENUE: 86931


In [12]:
folium.LayerControl().add_to(mapa)